# Sensitivity Analysis for Dynamic Pricing Engine

This notebook addresses the demand proxy's `elasticity_k` parameter and explores its impact on the optimal price and expected revenue.

## Section 1: Optimal Price vs Elasticity_k

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '..')
import config
from src.model import load_artifacts
from src.elasticity import build_demand_curve
from src.optimizer import find_optimal_price
from src.features import build_inference_row

plt.style.use('dark_background')

# Load artifacts and data
model, encoders, category_stats = load_artifacts()
train_df = pd.read_parquet('../' + str(config.PROCESSED_TRAIN))

# Pick 5 representative items (using dummy names and varied stats)
items = [
    {"item_name": "iPhone 12", "category_main": "Electronics", "category_sub": "Cell Phones", "category_leaf": "Smartphones", "brand_name": "Apple", "item_condition_id": 2, "shipping": 0, "item_description": "Great condition"},
    {"item_name": "Nike Hoodie", "category_main": "Men", "category_sub": "Sweats", "category_leaf": "Hoodies", "brand_name": "Nike", "item_condition_id": 1, "shipping": 1, "item_description": "Brand new"},
    {"item_name": "Harry Potter Book", "category_main": "Other", "category_sub": "Books", "category_leaf": "Fiction", "brand_name": "unknown", "item_condition_id": 3, "shipping": 0, "item_description": "Read once"},
    {"item_name": "Lego Set", "category_main": "Kids", "category_sub": "Toys", "category_leaf": "Building Toys", "brand_name": "Lego", "item_condition_id": 1, "shipping": 1, "item_description": "Sealed box"},
    {"item_name": "Vintage Watch", "category_main": "Vintage & Collectibles", "category_sub": "Antique", "category_leaf": "Jewelry", "brand_name": "unknown", "item_condition_id": 4, "shipping": 0, "item_description": "Needs repair"}
]

# Build feature rows
features_list = []
for item in items:
    X = build_inference_row(item, encoders, category_stats)
    feat_dict = {k: float(X[0, i]) for i, k in enumerate(config.ALL_FEATURES)}
    features_list.append((item['item_name'], feat_dict))


In [ ]:
k_values = np.linspace(0.3, 3.0, 20)

plt.figure(figsize=(10, 6))

results_data = {name: {'prices': [], 'revenues': []} for name, _ in features_list}

for name, feat_dict in features_list:
    cat_median = feat_dict.get("category_price_median", 25.0)
    cat_std = feat_dict.get("category_price_std", 15.0)
    
    for k in k_values:
        curve = build_demand_curve(feat_dict, model, cat_median, cat_std, elasticity_k=k)
        res = find_optimal_price(curve)
        results_data[name]['prices'].append(res['optimal_price'])
        results_data[name]['revenues'].append(res['optimal_revenue'])
        
    plt.plot(k_values, results_data[name]['prices'], marker='o', label=name)

plt.title('Optimal Price vs Elasticity Assumption (k)')
plt.xlabel('elasticity_k')
plt.ylabel('Optimal Price ($)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**Interpretation:** If the lines are flat, the optimizer is robust to the exact elasticity assumption. If they're steep, the elasticity parameter drastically changes the pricing recommendation, meaning it must be tuned carefully.

## Section 2: Revenue Sensitivity

In [ ]:
plt.figure(figsize=(10, 6))

for name, data in results_data.items():
    revs = np.array(data['revenues'])
    base_rev = revs[np.argmin(np.abs(k_values - 1.0))] # Revenue at k=1.0
    
    plt.plot(k_values, revs, marker='o', label=name)
    plt.fill_between(k_values, base_rev * 0.9, base_rev * 1.1, alpha=0.05, color='gray')

plt.title('Expected Revenue vs Elasticity Assumption (k)')
plt.xlabel('elasticity_k')
plt.ylabel('Expected Revenue ($)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**Interpretation:** The shaded regions represent a ±10% margin around the k=1.0 baseline. This shows the range of potential outcomes and how sensitive our projected yields are to the assumed demand decay rate.

## Section 3: Price Sweep Visualization

In [ ]:
# Pick the first item
name, feat_dict = features_list[0]
cat_median = feat_dict.get("category_price_median", 25.0)
cat_std = feat_dict.get("category_price_std", 15.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

for k, color in [(0.5, 'cyan'), (1.0, 'lime'), (2.0, 'magenta')]:
    curve = build_demand_curve(feat_dict, model, cat_median, cat_std, elasticity_k=k)
    res = find_optimal_price(curve)
    
    ax1.plot(curve['prices'], curve['demands'], color=color, label=f'k={k}')
    
    ax2.plot(curve['prices'], curve['revenues'], color=color, label=f'k={k}')
    ax2.plot(res['optimal_price'], res['optimal_revenue'], marker='*', color='white', markersize=15)

ax1.set_title(f'Demand Decay Curves ({name})')
ax1.set_xlabel('Price ($)')
ax1.set_ylabel('Demand Multiplier')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.set_title(f'Revenue Curves ({name})')
ax2.set_xlabel('Price ($)')
ax2.set_ylabel('Expected Revenue ($)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Conclusion

The `elasticity_k` parameter represents the sensitivity of demand to price deviations from the model's predicted "fair market value". A higher `k` means demand drops off sharply when prices are raised, while a lower `k` implies a more inelastic market where buyers are willing to pay a premium. 

Our sensitivity analysis reveals whether the optimizer is robust to this assumption. For many items, the optimal price is relatively stable across a range of `k` values, but for others, the revenue peak shifts dramatically. 

To estimate `k` empirically, we would need actual historical sales data showing the conversion rate (sell-through probability) at various price points relative to the market baseline. A controlled A/B test exposing different prices to different user cohorts would provide the definitive ground truth needed to calibrate this parameter.
